# MongoDB Tutorial with Python using the Netflix Dataset
Welcome! This notebook will guide you through using MongoDB with Python, leveraging the Netflix Movies and TV Shows dataset. You'll learn how to connect to MongoDB, load data, perform queries, aggregations, updates, deletions, and indexing.

We use the Netflix Movies and TV Shows dataset to demonstrate various MongoDB operations. This tutorial is suitable for beginners and intermediate users who want hands-on experience with MongoDB and Python.

## 1. Setup
Let's start by installing the required Python libraries. `pymongo` allows Python to interact with MongoDB, and `pandas` is used for data manipulation and analysis.

In the Docker container,  and  are preinstalled via . If you run this notebook in a local Python environment, you can install them using pip below:

In [1]:
# !pip install pymongo pandas

Now, let's import the libraries so we can use them in the notebook.

In [2]:
import pandas as pd
from pymongo import MongoClient
import json

## 2. Connecting to MongoDB
We will connect to a MongoDB instance running in Docker. The connection string includes the username (), password (), and host ( within the Docker network, or  if running outside the container).

Make sure your MongoDB container is running via .

The following code establishes a connection to MongoDB and selects the `netflix` database.

In [3]:
import os

mongo_host = os.getenv('MONGO_HOST', 'mongodb')

try:
    client = MongoClient(f'mongodb://root:example@{mongo_host}:27017/', serverSelectionTimeoutMS=2000)
    client.server_info()
except Exception:
    # Fallback to localhost if executed outside the Docker network
    mongo_host = 'localhost'
    client = MongoClient(f'mongodb://root:example@{mongo_host}:27017/')

db = client['netflix']
print(f"Connected to MongoDB at '{mongo_host}:27017'")

Connected to MongoDB at 'mongodb:27017'


## 3. Data Loading and Preparation
Let's load the Netflix dataset from a CSV file into a pandas DataFrame. We'll then convert the DataFrame into a list of dictionaries, which is the format MongoDB expects for bulk inserts.

The dataset  is mounted under  inside the Jupyter container (or  relative to this notebook).

In [4]:
import os

csv_candidates = [
    '/app/datasets/netflix_titles.csv',
    '../datasets/netflix_titles.csv',
    'datasets/netflix_titles.csv',
    'netflix_titles.csv'
]
dataset_path = next((p for p in csv_candidates if os.path.exists(p)), 'netflix_titles.csv')

df = pd.read_csv(dataset_path)
records = json.loads(df.to_json(orient='records'))
print(f"Loaded {len(records)} records from '{dataset_path}'")

Loaded 8807 records from '/app/datasets/netflix_titles.csv'


## 4. Database and Collection
Now, we'll create a collection named `titles` in the `netflix` database and insert all the records from the dataset.

The following code creates the collection and inserts the data. If the collection already exists, MongoDB will use the existing one.

In [5]:
collection = db['titles']
collection.insert_many(records)

InsertManyResult([ObjectId('6ac6340f7a022c5fcde966fa'), ObjectId('6ac6340f7a022c5fcde966fb'), ObjectId('6ac6340f7a022c5fcde966fc'), ObjectId('6ac6340f7a022c5fcde966fd'), ObjectId('6ac6340f7a022c5fcde966fe'), ObjectId('6ac6340f7a022c5fcde966ff'), ObjectId('6ac6340f7a022c5fcde96700'), ObjectId('6ac6340f7a022c5fcde96701'), ObjectId('6ac6340f7a022c5fcde96702'), ObjectId('6ac6340f7a022c5fcde96703'), ObjectId('6ac6340f7a022c5fcde96704'), ObjectId('6ac6340f7a022c5fcde96705'), ObjectId('6ac6340f7a022c5fcde96706'), ObjectId('6ac6340f7a022c5fcde96707'), ObjectId('6ac6340f7a022c5fcde96708'), ObjectId('6ac6340f7a022c5fcde96709'), ObjectId('6ac6340f7a022c5fcde9670a'), ObjectId('6ac6340f7a022c5fcde9670b'), ObjectId('6ac6340f7a022c5fcde9670c'), ObjectId('6ac6340f7a022c5fcde9670d'), ObjectId('6ac6340f7a022c5fcde9670e'), ObjectId('6ac6340f7a022c5fcde9670f'), ObjectId('6ac6340f7a022c5fcde96710'), ObjectId('6ac6340f7a022c5fcde96711'), ObjectId('6ac6340f7a022c5fcde96712'), ObjectId('6ac6340f7a022c5fcde967

## 5. Querying Data
Let's explore how to query data from MongoDB. We'll start with basic queries and move to more specific ones.

### Find all documents
This query retrieves all documents in the collection, but we limit the output to just one for brevity.

In [6]:
for title in collection.find().limit(1): 
    print(title)

{'_id': ObjectId('6ac6340f7a022c5fcde966fa'), 'show_id': 's1', 'type': 'Movie', 'title': 'Dick Johnson Is Dead', 'director': 'Kirsten Johnson', 'cast': None, 'country': 'United States', 'date_added': 'September 25, 2021', 'release_year': 2020, 'rating': 'PG-13', 'duration': '90 min', 'listed_in': 'Documentaries', 'description': 'As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.'}


### Find movies directed by a specific director
This query finds all movies directed by 'Martin Scorsese'. You can change the director's name to search for others.

In [7]:
for title in collection.find({'director': 'Martin Scorsese'}):
    print(title['title'])

Shutter Island
No Direction Home: Bob Dylan
Hugo
The Irishman
Rolling Thunder Revue: A Bob Dylan Story by Martin Scorsese
Alice Doesn't Live Here Anymore
Gangs of New York
GoodFellas
Mean Streets
Raging Bull
The Departed
Who's That Knocking at My Door?


### Find TV Shows with a specific rating
This query finds TV Shows with the rating 'TV-MA'. The `.limit(5)` method restricts the output to five results.

In [8]:
for title in collection.find({'type': 'TV Show', 'rating': 'TV-MA'}).limit(5):
    print(title['title'])

Blood & Water
Ganglands
Jailbirds New Orleans
Kota Factory
Midnight Mass


### Find movies released after a certain year
This query finds movies released after 2020. You can change the year to suit your needs.

In [9]:
for title in collection.find({'type': 'Movie', 'release_year': {'$gt': 2020}}).limit(5):
    print(title['title'], title['release_year'])

My Little Pony: A New Generation 2021
The Starling 2021
Je Suis Karl 2021
Confessions of an Invisible Girl 2021
Intrusion 2021


## 6. Aggregation Framework
MongoDB's aggregation framework allows for advanced data analysis, such as grouping and sorting.

### Count the number of movies and TV shows
This aggregation groups documents by their `type` (Movie or TV Show) and counts how many of each exist.

In [10]:
pipeline = [
    {
        '$group': {
            '_id': '$type',
            'count': {'$sum': 1}
        }
    }
]
for result in collection.aggregate(pipeline):
    print(result)

{'_id': 'Movie', 'count': 6131}
{'_id': 'TV Show', 'count': 2676}


### Find the top 10 countries with the most content
This aggregation groups by `country`, counts the number of titles per country, sorts them in descending order, and returns the top 10.

In [11]:
pipeline = [
    {
        '$group': {
            '_id': '$country',
            'count': {'$sum': 1}
        }
    },
    {
        '$sort': {'count': -1}
    },
    {
        '$limit': 10
    }
]
for result in collection.aggregate(pipeline):
    print(result)

{'_id': 'United States', 'count': 2818}
{'_id': 'India', 'count': 972}
{'_id': None, 'count': 831}
{'_id': 'United Kingdom', 'count': 419}
{'_id': 'Japan', 'count': 245}
{'_id': 'South Korea', 'count': 199}
{'_id': 'Canada', 'count': 181}
{'_id': 'Spain', 'count': 145}
{'_id': 'France', 'count': 124}
{'_id': 'Mexico', 'count': 110}


## 7. Updating Documents
You can update documents in MongoDB. The following example updates the rating of a specific title.

In [12]:
# Update the rating of a specific title
collection.update_one({'title': 'Dick Johnson Is Dead'}, {'$set': {'rating': 'PG-13 - Updated'}})
for title in collection.find({'title': 'Dick Johnson Is Dead'}):
    print(title['rating'])

PG-13 - Updated


## 8. Deleting Documents
You can delete documents from a collection. The following example deletes a title and verifies the deletion.

In [13]:
# Delete a title from the collection
collection.delete_one({'title': 'Dick Johnson Is Dead'})
print(collection.count_documents({'title': 'Dick Johnson Is Dead'}))

0


## 9. Indexing and Performance

### Performance without Index
Let's see how long it takes to query for movies released in a specific year without an index on the `release_year` field. We will use the `explain()` method to get detailed execution statistics.

In [14]:
print(collection.find({'release_year': 2020}).explain()['executionStats'])

{'executionSuccess': True, 'nReturned': 952, 'executionTimeMillis': 3, 'executionTimeMicros': 3011, 'totalKeysExamined': 0, 'totalDocsExamined': 8806, 'executionStages': {'isCached': False, 'stage': 'COLLSCAN', 'filter': {'release_year': {'$eq': 2020}}, 'nReturned': 952, 'executionTimeMillisEstimate': 0, 'works': 8807, 'advanced': 952, 'needTime': 7854, 'needYield': 0, 'saveState': 0, 'restoreState': 0, 'isEOF': 1, 'nss': 'netflix.titles', 'direction': 'forward', 'docsExamined': 8806, 'seeks': 0}, 'allPlansExecution': []}


### Creating an Index
Now, let's create an index on the `release_year` field. This will improve the performance of queries that filter by this field.

In [15]:
collection.create_index('release_year')

'release_year_1'

### Performance with Index
Let's run the same query again and see the difference in performance. You should see a significant decrease in `totalDocsExamined` and `executionTimeMillis`.

In [16]:
print(collection.find({'release_year': 2020}).explain()['executionStats'])

{'executionSuccess': True, 'nReturned': 952, 'executionTimeMillis': 1, 'executionTimeMicros': 1257, 'totalKeysExamined': 952, 'totalDocsExamined': 952, 'executionStages': {'isCached': False, 'stage': 'FETCH', 'nReturned': 952, 'executionTimeMillisEstimate': 0, 'works': 953, 'advanced': 952, 'needTime': 0, 'needYield': 0, 'saveState': 0, 'restoreState': 0, 'isEOF': 1, 'nss': 'netflix.titles', 'docsExamined': 952, 'alreadyHasObj': 0, 'inputStage': {'stage': 'IXSCAN', 'nReturned': 952, 'executionTimeMillisEstimate': 0, 'works': 953, 'advanced': 952, 'needTime': 0, 'needYield': 0, 'saveState': 0, 'restoreState': 0, 'isEOF': 1, 'nss': 'netflix.titles', 'keyPattern': {'release_year': 1}, 'indexName': 'release_year_1', 'isMultiKey': False, 'multiKeyPaths': {'release_year': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'release_year': ['[2020, 2020]']}, 'keysExamined': 952, 'seeks': 1, 'dupsTested': 0, 'dupsDropped': 0

### Existing Indexing Example

Indexes improve query performance. The following example creates an index on the `title` field and displays all indexes for the collection.

In [17]:
# Create an index on the 'title' field
collection.create_index('title')
print(collection.index_information())

{'_id_': {'v': 2, 'key': [('_id', 1)], 'collation': SON([('locale', 'simple')])}, 'release_year_1': {'v': 2, 'key': [('release_year', 1)], 'collation': SON([('locale', 'simple')])}, 'title_1': {'v': 2, 'key': [('title', 1)], 'collation': SON([('locale', 'simple')])}}
